# Notebook 5: Decision Tree — Brain Stroke Prediction

## Overview
A **Decision Tree** is a tree-structured classifier that splits the data on feature thresholds, making it easy to visualise and interpret.

Key hyperparameters:
* `max_depth` – limits tree depth to prevent overfitting.
* `min_samples_split` – minimum samples required to split a node.
* `criterion` – impurity measure (`gini` or `entropy`).

## Dataset — Stroke Prediction Dataset (Kaggle)
**Source:** https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset

Contains 5,110 patient records with attributes such as age, hypertension, heart disease, BMI, etc., and a binary target indicating whether the patient had a stroke.

### How to Download
```bash
pip install kaggle
# Place kaggle.json at ~/.kaggle/kaggle.json
kaggle datasets download -d fedesoriano/stroke-prediction-dataset \
  -p data/stroke --unzip
```

In [ ]:
# ─── Install (uncomment if needed) ─────────────────────────────────────────
# !pip install kaggle pandas scikit-learn matplotlib seaborn imbalanced-learn

# ─── Download dataset ───────────────────────────────────────────────────────
# !kaggle datasets download -d fedesoriano/stroke-prediction-dataset \
#   -p data/stroke --unzip

## 1. Imports and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────
DATA_PATH = 'data/stroke/healthcare-dataset-stroke-data.csv'

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Kaggle dataset loaded: {df.shape}")
except FileNotFoundError:
    print("Kaggle file not found – building synthetic brain-disease-like dataset.")
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'age':              np.random.randint(20, 85, n),
        'hypertension':     np.random.randint(0, 2, n),
        'heart_disease':    np.random.randint(0, 2, n),
        'avg_glucose_level':np.random.uniform(50, 280, n),
        'bmi':              np.random.uniform(15, 50, n),
        'gender':           np.random.choice(['Male','Female'], n),
        'ever_married':     np.random.choice(['Yes','No'], n),
        'work_type':        np.random.choice(['Private','Self-employed','Govt_job','children'], n),
        'Residence_type':   np.random.choice(['Urban','Rural'], n),
        'smoking_status':   np.random.choice(['formerly smoked','never smoked','smokes','Unknown'], n),
    })
    # Construct a roughly realistic target
    stroke_prob = (0.3 * (df['age'] > 60).astype(int)
                 + 0.2 * df['hypertension']
                 + 0.2 * df['heart_disease']
                 + 0.1 * (df['avg_glucose_level'] > 150).astype(int))
    df['stroke'] = (stroke_prob + np.random.uniform(0, 0.3, n) > 0.45).astype(int)
    print(f"Synthetic dataset created: {df.shape}")

df.head()

## 2. Exploratory Data Analysis

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget (stroke) distribution:")
print(df['stroke'].value_counts())
print(f"  Class imbalance: {df['stroke'].mean()*100:.1f}% stroke cases")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Target distribution
df['stroke'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','salmon'])
axes[0].set_title('Stroke Distribution (0=No, 1=Yes)')
axes[0].set_xlabel('Stroke'); axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Age distribution by stroke
df[df['stroke']==0]['age'].hist(bins=20, alpha=0.6, ax=axes[1], label='No Stroke', color='steelblue')
df[df['stroke']==1]['age'].hist(bins=20, alpha=0.6, ax=axes[1], label='Stroke',    color='salmon')
axes[1].set_title('Age Distribution by Stroke')
axes[1].set_xlabel('Age'); axes[1].legend()

# Glucose level distribution by stroke
df[df['stroke']==0]['avg_glucose_level'].hist(bins=20, alpha=0.6, ax=axes[2], label='No Stroke', color='steelblue')
df[df['stroke']==1]['avg_glucose_level'].hist(bins=20, alpha=0.6, ax=axes[2], label='Stroke',    color='salmon')
axes[2].set_title('Glucose Level by Stroke')
axes[2].set_xlabel('Avg Glucose Level'); axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Drop ID column if present
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)

# Fill missing BMI with median
if 'bmi' in df.columns:
    df['bmi'].fillna(df['bmi'].median(), inplace=True)

# Encode categorical features
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# Separate features and target
X = df.drop(columns=['stroke'])
y = df['stroke']

print(f"\nFeatures shape : {X.shape}")
print(f"Target shape   : {y.shape}")

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

## 4. Train Decision Tree

In [ ]:
# Start with a shallow tree for interpretability
dtree = DecisionTreeClassifier(
    max_depth=5,
    criterion='gini',
    min_samples_split=10,
    random_state=42
)
dtree.fit(X_train, y_train)

y_pred = dtree.predict(X_test)

print(f"Train Accuracy : {dtree.score(X_train, y_train):.4f}")
print(f"Test Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC        : {roc_auc_score(y_test, dtree.predict_proba(X_test)[:, 1]):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Stroke', 'Stroke']))

## 5. Visualise the Decision Tree

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(
    dtree,
    feature_names=X.columns.tolist(),
    class_names=['No Stroke', 'Stroke'],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title('Decision Tree – Brain Stroke Prediction (max_depth=5)')
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
feat_imp = pd.Series(dtree.feature_importances_, index=X.columns)\
             .sort_values(ascending=False)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='bar', color='steelblue')
plt.title('Feature Importance – Decision Tree')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()

print(feat_imp)

## 7. Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No Stroke', 'Stroke'],
    cmap='Blues'
)
plt.title('Decision Tree — Confusion Matrix')
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'max_depth':        [3, 5, 7, 10, None],
    'criterion':        ['gini', 'entropy'],
    'min_samples_split':[5, 10, 20],
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV ROC-AUC : {grid_search.best_score_:.4f}")

best_dtree   = grid_search.best_estimator_
best_y_pred  = best_dtree.predict(X_test)
best_auc     = roc_auc_score(y_test, best_dtree.predict_proba(X_test)[:, 1])
print(f"Tuned Test Accuracy : {accuracy_score(y_test, best_y_pred):.4f}")
print(f"Tuned Test ROC-AUC  : {best_auc:.4f}")

## 9. Effect of max_depth on Accuracy

In [ ]:
depths = range(1, 16)
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_scores, marker='o', label='Train Accuracy')
plt.plot(depths, test_scores,  marker='s', label='Test Accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree Depth vs Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

## Summary

* Decision Trees are **interpretable** — the tree diagram explains every decision.
* Without pruning (`max_depth`), trees overfit the training data.
* The **Gini impurity** and **entropy** criteria give similar results.
* Important stroke predictors: **age**, **avg_glucose_level**, and **bmi**.
* For imbalanced datasets (stroke is rare), **ROC-AUC** is a better metric than accuracy.